# 02 - Kaggle Supervised Baseline Runner

This notebook runs the four supervised baseline experiments on Kaggle GPU:

- `resnet18_none`
- `resnet18_imagenet`
- `vit_s16_none`
- `vit_s16_imagenet`

The notebook only prepares the Kaggle environment and calls repository scripts. It does not contain model training logic.

## 1. Enable Kaggle GPU

Before running, open **Settings** in Kaggle and select a GPU accelerator. T4 is enough for these baseline runs.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import pandas as pd

print('Python:', sys.version)
print('Working directory:', Path.cwd())

## 2. Clone Or Pull Repository

This keeps the Kaggle working directory synchronized with the GitHub repository.

In [ ]:
REPO_URL = 'https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git'
REPO_ROOT = Path('/kaggle/working/contrastive-synthesis-medcls_CVProject')

if REPO_ROOT.exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)

os.chdir(REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT)

## 3. Install Minimal Dependencies

Kaggle already provides PyTorch. This cell installs only the packages commonly needed by the training/evaluation scripts.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'timm', 'scikit-learn', 'matplotlib', 'pandas', 'Pillow', 'pyyaml'
], check=True)

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 4. Editable Runner Variables

Set the Kaggle dataset path and choose which baseline experiments to run. Add your Kaggle dataset named `medcls-cvproject` from the right panel before running data checks.

In [ ]:
# Kaggle dataset should contain:
# /kaggle/input/medcls-cvproject/data/processed/labelled_4232
# /kaggle/input/medcls-cvproject/data/manifests/train.csv, val.csv, test.csv
DATA_ROOT = Path('/kaggle/input/medcls-cvproject/data')
LABELLED_SOURCE = DATA_ROOT / 'processed/labelled_4232'
MANIFEST_SOURCE = DATA_ROOT / 'manifests'

OUTPUT_ROOT = Path('/kaggle/working/results/experiments')

RUN_RESNET_NONE = True
RUN_RESNET_IMAGENET = True
RUN_VIT_NONE = True
RUN_VIT_IMAGENET = True

# Use 1 for smoke test. Use None for config/default epochs.
EPOCH_OVERRIDE = None
NUM_WORKERS = 2

print('DATA_ROOT:', DATA_ROOT)
print('LABELLED_SOURCE exists:', LABELLED_SOURCE.exists())
print('MANIFEST_SOURCE exists:', MANIFEST_SOURCE.exists())
print('OUTPUT_ROOT:', OUTPUT_ROOT)

## 5. Link Data And Manifests

The repo scripts expect data under `data/processed` and manifests under `data/manifests`. This cell symlinks Kaggle input data into the cloned repository when available.

In [ ]:
def replace_path(target: Path, source: Path):
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() or target.is_symlink():
        if target.is_symlink() or target.is_file():
            target.unlink()
        else:
            shutil.rmtree(target)
    if source.exists():
        os.symlink(source, target, target_is_directory=source.is_dir())
        print('Linked:', target, '->', source)
    else:
        print('WARNING: source missing:', source)

replace_path(REPO_ROOT / 'data/processed/labelled_4232', LABELLED_SOURCE)

(REPO_ROOT / 'data/manifests').mkdir(parents=True, exist_ok=True)
for name in ['train.csv', 'val.csv', 'test.csv', 'labelled_all.csv', 'split_summary.json']:
    src = MANIFEST_SOURCE / name
    dst = REPO_ROOT / 'data/manifests' / name
    if src.exists():
        if dst.exists() or dst.is_symlink():
            dst.unlink()
        os.symlink(src, dst)
        print('Linked manifest:', dst, '->', src)
    elif dst.exists():
        print('Using repo manifest:', dst)
    else:
        print('WARNING: missing manifest:', src)

print('
Repository data snapshot:')
subprocess.run(['find', 'data', '-maxdepth', '3', '-type', 'd'], check=False)
subprocess.run(['ls', '-lh', 'data/manifests'], check=False)

## 6. Lightweight Checks

These checks should pass before launching training.

In [ ]:
subprocess.run([sys.executable, 'scripts/check_experiment_inputs.py'], check=True)
subprocess.run([
    sys.executable, '-m', 'py_compile',
    'scripts/run_classification_resnet.py',
    'scripts/evaluate_classification_resnet.py',
    'scripts/run_classification_vit.py',
    'scripts/evaluate_classification_vit.py',
], check=True)

## 7. Helper Function

In [ ]:
def run_cmd(cmd):
    print('Running:')
    print(' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True)

def epoch_args():
    return [] if EPOCH_OVERRIDE is None else ['--epochs', str(EPOCH_OVERRIDE)]

def show_metrics(experiment_id):
    metrics_path = OUTPUT_ROOT / experiment_id / 'metrics.json'
    if not metrics_path.exists():
        print('Missing metrics:', metrics_path)
        return None
    metrics = json.loads(metrics_path.read_text())
    row = {'experiment_id': experiment_id, **metrics}
    display(pd.DataFrame([row]))
    return row

## 8. Experiment: resnet18_none

Train ResNet18 from random initialization on the fixed real labeled train/val/test manifests.

In [ ]:
EXP = 'resnet18_none'
if RUN_RESNET_NONE:
    run_cmd([
        sys.executable, 'scripts/run_classification_resnet.py',
        '--config', 'configs/experiments/resnet18/none.yaml',
        '--manifest-dir', 'data/manifests',
        '--output-dir', OUTPUT_ROOT / EXP,
        '--num-workers', NUM_WORKERS,
        *epoch_args(),
    ])
    show_metrics(EXP)
else:
    print('Skipping', EXP)

## 9. Experiment: resnet18_imagenet

Fine-tune ImageNet-pretrained ResNet18 on the fixed real labeled train/val/test manifests.

In [ ]:
EXP = 'resnet18_imagenet'
if RUN_RESNET_IMAGENET:
    run_cmd([
        sys.executable, 'scripts/run_classification_resnet.py',
        '--config', 'configs/experiments/resnet18/imagenet.yaml',
        '--manifest-dir', 'data/manifests',
        '--output-dir', OUTPUT_ROOT / EXP,
        '--num-workers', NUM_WORKERS,
        *epoch_args(),
    ])
    show_metrics(EXP)
else:
    print('Skipping', EXP)

## 10. Experiment: vit_s16_none

Train ViT-S/16 from random initialization on the fixed real labeled train/val/test manifests.

In [ ]:
EXP = 'vit_s16_none'
if RUN_VIT_NONE:
    run_cmd([
        sys.executable, 'scripts/run_classification_vit.py',
        '--config', 'configs/experiments/vit_s16/none.yaml',
        '--manifest-dir', 'data/manifests',
        '--output-dir', OUTPUT_ROOT / EXP,
        '--num-workers', NUM_WORKERS,
        *epoch_args(),
    ])
    show_metrics(EXP)
else:
    print('Skipping', EXP)

## 11. Experiment: vit_s16_imagenet

Fine-tune ImageNet-pretrained ViT-S/16 on the fixed real labeled train/val/test manifests.

In [ ]:
EXP = 'vit_s16_imagenet'
if RUN_VIT_IMAGENET:
    run_cmd([
        sys.executable, 'scripts/run_classification_vit.py',
        '--config', 'configs/experiments/vit_s16/imagenet.yaml',
        '--manifest-dir', 'data/manifests',
        '--output-dir', OUTPUT_ROOT / EXP,
        '--num-workers', NUM_WORKERS,
        *epoch_args(),
    ])
    show_metrics(EXP)
else:
    print('Skipping', EXP)

## 12. Summary Table

In [ ]:
rows = []
for exp in ['resnet18_none', 'resnet18_imagenet', 'vit_s16_none', 'vit_s16_imagenet']:
    metrics_path = OUTPUT_ROOT / exp / 'metrics.json'
    if metrics_path.exists():
        metrics = json.loads(metrics_path.read_text())
        rows.append({'experiment_id': exp, **metrics})

summary = pd.DataFrame(rows)
if len(summary):
    display(summary)
    summary_path = OUTPUT_ROOT / 'supervised_baseline_summary.csv'
    summary.to_csv(summary_path, index=False)
    print('Saved:', summary_path)
else:
    print('No completed baseline metrics found yet.')

## 13. Package Results

Kaggle output files can be downloaded from `/kaggle/working`. This cell creates one zip archive for the baseline outputs.

In [ ]:
zip_base = Path('/kaggle/working/supervised_baseline_results')
if zip_base.with_suffix('.zip').exists():
    zip_base.with_suffix('.zip').unlink()
shutil.make_archive(str(zip_base), 'zip', root_dir=OUTPUT_ROOT.parent, base_dir='experiments')
print('Created:', zip_base.with_suffix('.zip'))